# 03 — Data Cleaning
**ChurnOps Phase 4** — Deterministic, Domain-Verified Pre-Split Cleaning  
*All cleaning operations are driven directly by EDA/Validation findings without statistical data leakage.*

---

### Phase 4 Objectives
1. **TotalCharges Resolution:** Impute `0.0` for the 11 records with whitespace strings (verified in Phase 3 to have `tenure == 0`) and cast to `float64`.
2. **Whitespace Normalization:** Strip extraneous whitespace from all text/categorical features.
3. **Target Encoding:** Map categorical `Churn` (`{'No', 'Yes'}`) to integer binary (`{0, 1}`).
4. **Identifier Elimination:** Drop `customerID` (unique identifier with 100% cardinality, zero generalizable signal).
5. **Leakage Boundary Check:** Ensure no transformations *learn* statistics from the data (e.g. median/mean imputation or scaling, which must occur within CV folds in Phase 8).
6. **Artifact Output:** Persist clean dataset to `data/interim/telco_clean.csv` and emit `reports/cleaning_log.json`.

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.utils.config import get_path, PARAMS
from src.data.cleaning import clean_dataset, run_cleaning_pipeline

print("Project Root:", PROJECT_ROOT)

Project Root: C:\Users\Abhishek\OneDrive\Pictures\Desktop\ChurnOps


In [2]:
raw_df = pd.read_csv(get_path('raw_data'))
print(f"Raw Data Shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]:,} columns")
raw_df.head(3)

Raw Data Shape: 7,043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 1. Execute Cleaning Transformations & Audit Log

In [3]:
clean_df, audit_log = clean_dataset(raw_df)
print(f"Cleaned Data Shape: {clean_df.shape[0]:,} rows x {clean_df.shape[1]:,} columns")
print(f"Total Transformations Applied: {len(audit_log['transformations'])}")

[2026-10-05 13:34:36] INFO     cleaning - Starting data cleaning on input shape: [7043, 21]


[2026-10-05 13:34:36] INFO     cleaning - Step 1 - Whitespace stripped on: {'TotalCharges': 11}


[2026-10-05 13:34:36] INFO     cleaning - Step 2 - TotalCharges fixed: 11 blank rows set to 0.0, converted to float64


[2026-10-05 13:34:36] INFO     cleaning - Step 3 - Target column 'Churn' mapped: {'No': 0, 'Yes': 1}


[2026-10-05 13:34:36] INFO     cleaning - Step 4 - Dropped identifier column 'customerID'


[2026-10-05 13:34:36] INFO     cleaning - Step 5 - Validation passed: shape=[7043, 20], 0 nulls, 0 duplicates


Cleaned Data Shape: 7,043 rows x 20 columns
Total Transformations Applied: 4


In [4]:
pd.DataFrame([
    {
        'Step': t['step'],
        'Name': t['name'],
        'Affected Rows': t['affected_rows'],
        'Rationale': t['rationale']
    }
    for t in audit_log['transformations']
])

,Step,Name,Affected Rows,Rationale
0,1,strip_whitespace,11,Removes invisible whitespace to prevent catego...
1,2,fix_totalcharges_blanks_and_type,11,Empirical evidence proves all 11 customers wit...
2,3,encode_target_binary,7043,Maps categorical target to standard binary int...
3,4,drop_identifier_column,7043,Unique alphanumeric customer identifier carrie...


## 2. Verification of TotalCharges Imputation & Type Fix

Let us verify the exact 11 records that previously contained whitespace strings:

In [5]:
blank_indices = raw_df[raw_df['TotalCharges'].str.strip() == ''].index

comparison = pd.DataFrame({
    'CustomerID (Raw)': raw_df.loc[blank_indices, 'customerID'],
    'Tenure': raw_df.loc[blank_indices, 'tenure'],
    'MonthlyCharges': raw_df.loc[blank_indices, 'MonthlyCharges'],
    'Raw TotalCharges': raw_df.loc[blank_indices, 'TotalCharges'],
    'Cleaned TotalCharges': clean_df.loc[blank_indices, 'TotalCharges']
})
comparison

,CustomerID (Raw),Tenure,MonthlyCharges,Raw TotalCharges,Cleaned TotalCharges
488,4472-LVYGI,0,52.55,,0.0
753,3115-CZMZD,0,20.25,,0.0
936,5709-LVOEQ,0,80.85,,0.0
1082,4367-NUYAO,0,25.75,,0.0
1340,1371-DWPAZ,0,56.05,,0.0
3331,7644-OMVMY,0,19.85,,0.0
3826,3213-VVOLG,0,25.35,,0.0
4380,2520-SGTTA,0,20.00,,0.0
5218,2923-ARZLG,0,19.70,,0.0
6670,4075-WKNIU,0,73.35,,0.0


## 3. Verification of Target Variable Encoding

In [6]:
print("Raw Churn Value Counts:")
print(raw_df['Churn'].value_counts())
print("\nCleaned Churn Value Counts:")
print(clean_df['Churn'].value_counts())

assert clean_df['Churn'].dtype == 'int64', "Churn dtype must be int64!"
assert clean_df['Churn'].sum() == (raw_df['Churn'] == 'Yes').sum(), "Churn counts do not match!"
print("\nTarget encoding successfully verified!")

Raw Churn Value Counts:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Cleaned Churn Value Counts:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target encoding successfully verified!


## 4. Final Integrity Checks on Interim Clean Dataset

In [7]:
print(f"Total Null Values in Cleaned Dataset: {clean_df.isnull().sum().sum()}")
print(f"Duplicate Rows in Cleaned Dataset     : {clean_df.duplicated().sum()}")
print(f"Is customerID column present?          : {'customerID' in clean_df.columns}")
print("\nCleaned Data Types:")
print(clean_df.dtypes)

Total Null Values in Cleaned Dataset: 0
Duplicate Rows in Cleaned Dataset     : 22
Is customerID column present?          : False

Cleaned Data Types:
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object


---

## 5. Phase 4 Summary & Hand-off to Phase 5
- **Input:** `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv` (7,043 rows x 21 cols, unchanged)
- **Output:** `data/interim/telco_clean.csv` (7,043 rows x 20 cols)
- **Missing Values:** 0
- **Leakage Status:** Fully compliant. All modifications are deterministic domain adjustments. No statistical parameters (means, medians, scalers) were fitted on the data.
- **Ready for:** Phase 5 — Feature Engineering.